# JSON preparation demo

Prepare condition-classification training and holdout JSONL from the bundled positive and negative records. The notebook calls the main dataset preparation module through `run_demo.py`. No API key or training job is needed.

Install from the repository root with `python -m pip install -e ".[curation,datasets,notebook]"`.

## Inputs and setup

Run the cells in order. The following paths are relative to `Demo/02_json_preparation/`; all required files are included.

| Input | Location | Contents |
| --- | --- | --- |
| Cleaned positives, CSV | `input/positive_cleaned.csv` | The expected stage 6 output from the cleaning demo. |
| Cleaned negatives, CSV | `input/negative_cleaned.csv` | Enumerated and curated negative-condition records. |
| Publication years, CSV | `input/publication_years.csv` | `DOI` and `Publication Year` columns. |
| Holdout conditions, JSON | `input/forced_questions.json` | Condition definitions selected by `config.json`. |
| Classification prompt, TXT | `../../prompts/training/reaction_prediction.txt` | The main workflow's classification prompt. |

To use the output of the cleaning demo, pass `positive_csv=REPO / "Demo/01_data_cleaning/outputs/mof_extraction_1_2_3_4_5_6.csv"` to `run_demo` below. To test different cleaned tables, select their paths in `config.json` and use `check=False`; `check=True` compares with the supplied dataset only. Keep the input column schemas and provide matching DOI/year metadata.

Outputs are saved together under `outputs/`: `mof_ft_train.jsonl`, `mof_ft_holdout.jsonl`, `mof_ft_split_assignments.csv`, and summary files. The split is computed locally; it does not start a training job.

Implementation: [demo runner](run_demo.py) and [dataset preparation](../../src/mofinder/datasets/prepare.py). See the [source-to-code guide](../../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import runpy
import pandas as pd

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "mofinder").is_dir():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from within the MOFinder repository.")

import json
DEMO = REPO / "Demo" / "02_json_preparation"


## 1. Inspect the inputs

In [ ]:
positive = pd.read_csv(DEMO / "input" / "positive_cleaned.csv")
negative = pd.read_csv(DEMO / "input" / "negative_cleaned.csv")
pd.DataFrame({"label": ["P", "N"], "input_rows": [len(positive), len(negative)]})


## 2. Prepare and verify the datasets

The seed-42 split groups records by metal precursor, linker set, and solvent set. The bundled positive input is the expected output of the cleaning demo.

In [ ]:
run_demo = runpy.run_path(str(DEMO / "run_demo.py"))["run"]
summary = run_demo(check=True)
pd.DataFrame({name: summary["labels"][name] for name in ["train", "holdout"]})


## 3. Compare a few holdout and training examples

Show **two P (success) and two N (failure) examples per split**, with holdout first. Change `N_PER_LABEL` below to display more or fewer examples. Within each split and label, examples are ordered by `source_row_id`.

The JSONL files are shuffled. The preview matches each assignment to its saved JSONL record by normalized reaction conditions and label, rather than assuming their row numbers agree. `jsonl_line` is the 1-based line number in that split's JSONL file.

In [ ]:
from collections import defaultdict
from html import escape
from IPython.display import HTML, Markdown, display
from mofinder.datasets.prepare import forced_question_condition_key

N_PER_LABEL = 2  # Up to two P and two N examples from each split.
if type(N_PER_LABEL) is not int or N_PER_LABEL < 1:
    raise ValueError("N_PER_LABEL must be a positive integer.")

assignments = pd.read_csv(
    DEMO / "outputs" / "mof_ft_split_assignments.csv",
    dtype={"is_success": "boolean", "publication_year": "Int64"},
)
assignments["label"] = assignments["is_success"].map({True: "P", False: "N"})

# Index the actual saved records; keep chemical spelling from the user-message JSON.
records_by_key = defaultdict(list)
for split in ["holdout", "train"]:
    with (DEMO / "outputs" / f"mof_ft_{split}.jsonl").open(encoding="utf-8-sig") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            conditions = json.loads(next(m["content"] for m in record["messages"] if m["role"] == "user"))
            label = next(m["content"] for m in record["messages"] if m["role"] == "assistant").strip()
            key = (split, forced_question_condition_key(conditions), label)
            records_by_key[key].append((line_number, conditions, record))

preview_rows = []
selected_examples = []
for split in ["holdout", "train"]:
    for label in ["P", "N"]:
        selected = assignments.loc[
            assignments["split"].eq(split) & assignments["label"].eq(label)
        ].sort_values("source_row_id").head(N_PER_LABEL)
        for row in selected.itertuples(index=False):
            key = (split, forced_question_condition_key(json.loads(row.condition_key)), label)
            matches = records_by_key.get(key, [])
            if len(matches) != 1:
                raise ValueError(f"Expected one JSONL match for source row {row.source_row_id}; found {len(matches)}.")
            line_number, conditions, record = matches[0]
            metadata = {
                "split": split, "source_row_id": row.source_row_id,
                "doi_norm": row.doi_norm, "publication_year": row.publication_year,
                "is_success": row.is_success, "label": label, "jsonl_line": line_number,
            }
            preview_rows.append(metadata)
            selected_examples.append({**metadata, "conditions": conditions, "record": record})

preview = pd.DataFrame(preview_rows)
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(preview)

## 4. Inspect reaction parameters and the full JSON records

Each column below is one of the selected examples. The eight parameters come from its saved user message, with units in the field names. Expand a record to see the complete system/user/assistant JSON. The assistant P/N value is the dataset label, not a new model prediction.


In [ ]:
for split in ["holdout", "train"]:
    examples = [item for item in selected_examples if item["split"] == split]
    display(Markdown(f"### {'Holdout' if split == 'holdout' else 'Training'} examples"))
    if not examples:
        print("No examples in this split.")
        continue

    parameters = pd.DataFrame({
        f"Source {item['source_row_id']} | {item['label']}": item["conditions"]
        for item in examples
    })
    parameters.index.name = "Reaction parameter"
    # Wrap long reagent names and display all values without pandas truncation.
    display(parameters.style.set_properties(**{
        "text-align": "left", "white-space": "normal", "overflow-wrap": "anywhere",
    }).format(str, na_rep="null"))

    for index, item in enumerate(examples):
        title = (f"Source {item['source_row_id']} | {item['label']} | {item['doi_norm']} | "
                 f"mof_ft_{split}.jsonl, line {item['jsonl_line']}")
        opened = " open" if index == 0 else ""
        display(HTML(
            f"<details{opened}><summary>{escape(title)}</summary>"
            '<pre style="white-space:pre-wrap;overflow-wrap:anywhere">'
            + escape(json.dumps(item["record"], ensure_ascii=False, indent=2))
            + "</pre></details>"
        ))

The full split summary records input hashes, filtering counts, and preparation settings. The small demo datasets are separate from the full research training and validation files.